<a href="https://colab.research.google.com/github/Varshu1915/PD-Classification-EEG-VWE/blob/main/PD_Classification_with_EEG_VWE_Ensembling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install libraries and mount Google Drive
!pip install mne
!pip -q install mne PyWavelets scikit-learn scipy pandas matplotlib

import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import pywt
import mne

from scipy.signal import welch, stft
from scipy.stats import skew
from scipy.integrate import trapezoid

from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import LeaveOneGroupOut, GroupKFold
from sklearn.metrics import (
  accuracy_score,
  precision_score,
  recall_score,
  f1_score,
  roc_auc_score,
  confusion_matrix,
  classification_report,
  ConfusionMatrixDisplay
)

import matplotlib.pyplot as plt


In [ ]:
#Mount Drive
from google.colab import drive
drive.mount("/content/drive")
# Google Drive paths
DATA_ROOT = Path("/content/drive/MyDrive/RAW_EEG_2")

In [ ]:
#Step 1 — Load the CSV and identify the five feature families
import os
import re
import random
import warnings
import numpy as np
import pandas as pd
import tensorflow as tf

from scipy.optimize import minimize
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import LeaveOneGroupOut, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, roc_auc_score
)
from tensorflow.keras import Model, Input
from tensorflow.keras.layers import (
    Dense, Dropout, Conv1D, MaxPooling1D,
    BatchNormalization, LSTM, MultiHeadAttention,
    LayerNormalization, Add, GlobalAveragePooling1D,
    Reshape, Concatenate
)
from tensorflow.keras.callbacks import EarlyStopping

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)
warnings.filterwarnings("ignore")

CSV_PATH = (
    "/content/drive/MyDrive/EEG_PD_Project/"
    "GWO_RF_LOSO/all_eeg_features.csv"
)
OUTPUT_DIR = (
    "/content/drive/MyDrive/EEG_PD_Project/"
    "RF_CNNLSTM_MultimodalTransformer_Ensemble"
)
os.makedirs(OUTPUT_DIR, exist_ok=True)

df = pd.read_csv(CSV_PATH)
df = df.dropna(subset=["subject_id", "label"]).copy()
df["subject_id"] = df["subject_id"].astype(str)

def encode_label(v):
    if isinstance(v, (int, float, np.integer, np.floating)):
        if v in (0, 1):
            return int(v)
    s = str(v).strip().lower()
    if s in {"pd", "1", "parkinson", "parkinsons",
             "parkinson's disease", "patient"}:
        return 1
    if s in {"control", "healthy", "healthy control", "0"}:
        return 0
    raise ValueError(f"Unknown label: {v}")

df["label"] = df["label"].apply(encode_label).astype(int)

metadata = [
    "subject_id", "recording_id", "epoch_index", "label"
]
feature_df = df.drop(
    columns=[c for c in metadata if c in df.columns]
)
feature_df = feature_df.apply(pd.to_numeric, errors="coerce")
feature_df = feature_df.replace([np.inf, -np.inf], np.nan)
feature_df = feature_df.dropna(axis=1, how="all")

valid_rows = feature_df.notna().any(axis=1)
df = df.loc[valid_rows].reset_index(drop=True)
feature_df = feature_df.loc[valid_rows].reset_index(drop=True)

feature_names = feature_df.columns.tolist()
X_raw = feature_df.to_numpy(dtype=np.float32)
y = df["label"].to_numpy(dtype=int)
groups = df["subject_id"].to_numpy()

print("Epochs:", len(y))
print("Features:", len(feature_names))
print("Subjects:", len(np.unique(groups)))
print("Class counts:", np.unique(y, return_counts=True))

# Classify columns using the feature naming conventions.
# Adjust these patterns if your actual CSV uses different names.
def modality_of(name):
    s = name.lower()

    if ("spectrogram" in s or "stft" in s or "spectrogram_" in s):
        return "spectrogram"

    if ("wav_" in s or "wavelet" in s or "_wav" in s):
        return "wavelet"

    if (
        s.startswith("coh_") or
        "connectivity" in s or
        "correlation" in s or
        s.startswith("corr_")
    ):
        return "connectivity"

    if (
        any(t in s for t in [
            "_mean", "_std", "_var", "_skew", "_kurtosis",
            "hjorth_activity", "hjorth_mobility",
            "hjorth_complexity"
        ])
    ):
        return "temporal"

    if (
        re.search(r"_(delta|theta|alpha|beta|gamma)$", s) or
        "spec_entropy" in s or
        "dom_freq" in s or
        "spectral" in s or
        "band_power" in s or
        "psd" in s
    ):
        return "spectral"

    return "other"

families = [
    "spectral", "temporal", "wavelet",
    "connectivity", "spectrogram", "other"
]

column_groups = {
    family: [
        i for i, name in enumerate(feature_names)
        if modality_of(name) == family
    ]
    for family in families
}

print("\nDetected feature families:")
for family, idx in column_groups.items():
    print(f"{family:15s}: {len(idx)} features")

# At least two actual feature families should be available
active_families = [
    f for f in families if len(column_groups[f]) > 0
]
if len(active_families) < 2:
    raise ValueError(
        "The column names did not identify enough feature families. "
        "Inspect feature_names and update modality_of()."
    )

Epochs: 14714
Features: 3969
Subjects: 21
Class counts: (array([0, 1]), array([ 4006, 10708]))

Detected feature families:
spectral       : 315 features
temporal       : 0 features
wavelet        : 882 features
connectivity   : 1953 features
spectrogram    : 630 features
other          : 189 features


In [ ]:
#Step 2 — Define the three models
def build_cnn_lstm(input_shape):
    # input_shape must be (number_of_features, 1)
    inp = Input(shape=input_shape)

    x = Conv1D(32, 7, padding="same", activation="relu")(inp)
    x = BatchNormalization()(x)
    x = MaxPooling1D(4, padding="same")(x)
    x = Dropout(0.2)(x)

    x = Conv1D(64, 5, padding="same", activation="relu")(x)
    x = BatchNormalization()(x)
    x = MaxPooling1D(2, padding="same")(x)
    x = Dropout(0.25)(x)

    x = Conv1D(64, 3, padding="same", activation="relu")(x)
    x = BatchNormalization()(x)
    x = MaxPooling1D(2, padding="same")(x)

    x = LSTM(32, return_sequences=True)(x)
    x = LSTM(32)(x)

    x = Dense(32, activation="relu")(x)
    x = Dropout(0.3)(x)

    out = Dense(
        1, activation="sigmoid", dtype="float32"
    )(x)

    model = Model(inp, out, name="Hybrid_CNN_LSTM")

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model
def build_multimodal_transformer(input_dims):
    inputs = []
    tokens = []

    for i, dim in enumerate(input_dims):
        inp = Input(shape=(dim,), name=f"modality_{i}")
        inputs.append(inp)

        # Each feature family becomes one embedding token
        token = Dense(32, activation="relu")(inp)
        token = Reshape((1, 32))(token)
        tokens.append(token)

    x = Concatenate(axis=1)(tokens)

    attn = MultiHeadAttention(
        num_heads=4,
        key_dim=8,
        dropout=0.1
    )(x, x)

    x = Add()([x, attn])
    x = LayerNormalization()(x)

    ff = Dense(64, activation="relu")(x)
    ff = Dropout(0.2)(ff)
    ff = Dense(32)(ff)

    x = Add()([x, ff])
    x = LayerNormalization()(x)
    x = GlobalAveragePooling1D()(x)

    x = Dense(32, activation="relu")(x)
    x = Dropout(0.3)(x)
    out = Dense(1, activation="sigmoid", dtype="float32")(x)

    model = Model(inputs, out, name="Multimodal_Transformer")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(0.001),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )
    return model

In [ ]:
#Step 3 — Define ensemble functions
def learn_ensemble_weights(val_probs, y_val):
    # Columns: RF, CNN-LSTM, Transformer
    if len(np.unique(y_val)) < 2:
        return np.ones(3) / 3

    eps = 1e-7

    def loss_fn(w):
        p = np.clip(val_probs @ w, eps, 1 - eps)
        return -np.mean(
            y_val * np.log(p) +
            (1 - y_val) * np.log(1 - p)
        )

    result = minimize(
        loss_fn,
        x0=np.ones(3) / 3,
        method="SLSQP",
        bounds=[(0, 1)] * 3,
        constraints=[
            {"type": "eq", "fun": lambda w: np.sum(w) - 1}
        ],
        options={"maxiter": 100}
    )

    return result.x if result.success else np.ones(3) / 3


def get_metrics(y_true, prob):
    pred = (prob >= 0.5).astype(int)
    cm = confusion_matrix(y_true, pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    return {
        "accuracy": accuracy_score(y_true, pred),
        "precision": precision_score(
            y_true, pred, zero_division=0
        ),
        "sensitivity": recall_score(
            y_true, pred, zero_division=0
        ),
        "specificity": tn / (tn + fp) if tn + fp else 0.0,
        "f1_score": f1_score(
            y_true, pred, zero_division=0
        ),
        "roc_auc": (
            roc_auc_score(y_true, prob)
            if len(np.unique(y_true)) == 2 else np.nan
        ),
        "TN": tn, "FP": fp, "FN": fn, "TP": tp
    }

In [ ]:
#Step 4 — Run feature selection, LOSO and ensemble comparison
logo = LeaveOneGroupOut()

MAX_PER_FAMILY = 64
EPOCHS = 15
BATCH_SIZE = 64
PATIENCE = 3

all_predictions = []
fold_records = []
weight_records = []

for fold, (outer_train, test_idx) in enumerate(
    logo.split(X_raw, y, groups), start=1
):
    print(f"\n===== LOSO {fold}/{len(np.unique(groups))} =====")

    X_outer = X_raw[outer_train]
    y_outer = y[outer_train]
    g_outer = groups[outer_train]

    X_test_raw = X_raw[test_idx]
    y_test = y[test_idx]
    test_subject = groups[test_idx][0]

    # Group-wise validation split inside outer training
    splitter = GroupShuffleSplit(
        n_splits=1, test_size=0.20, random_state=SEED + fold
    )
    inner_idx, val_idx = next(
        splitter.split(X_outer, y_outer, g_outer)
    )

    X_inner_raw, y_inner = X_outer[inner_idx], y_outer[inner_idx]
    X_val_raw, y_val = X_outer[val_idx], y_outer[val_idx]

    if len(np.unique(y_inner)) < 2:
        raise ValueError(
            f"Fold {fold}: inner training split contains only one class. "
            "Use a different group-wise split strategy."
        )

    # Fit preprocessing only on inner training data
    imputer = SimpleImputer(strategy="median")
    Xi = imputer.fit_transform(X_inner_raw)
    Xv = imputer.transform(X_val_raw)
    Xt = imputer.transform(X_test_raw)

    scaler = StandardScaler()
    Xi = scaler.fit_transform(Xi)
    Xv = scaler.transform(Xv)
    Xt = scaler.transform(Xt)

    # Separate feature selection for each modality
    selected_groups = {}
    selected_names = {}
    selected_count = {}

    for family in active_families:
        cols = column_groups[family]

        Xi_family = Xi[:, cols]
        Xv_family = Xv[:, cols]
        Xt_family = Xt[:, cols]

        k = min(MAX_PER_FAMILY, len(cols))

        # SelectKBest requires at least one feature
        selector = SelectKBest(score_func=f_classif, k=k)
        Xi_family = selector.fit_transform(Xi_family, y_inner)
        Xv_family = selector.transform(Xv_family)
        Xt_family = selector.transform(Xt_family)

        selected_groups[family] = (Xi_family, Xv_family, Xt_family)
        selected_names[family] = [
            feature_names[cols[i]]
            for i, keep in enumerate(selector.get_support())
            if keep
        ]
        selected_count[family] = Xi_family.shape[1]

    # Concatenated selected features for RF and CNN-LSTM
    Xi_all = np.concatenate(
        [selected_groups[f][0] for f in active_families], axis=1
    )
    Xv_all = np.concatenate(
        [selected_groups[f][1] for f in active_families], axis=1
    )
    Xt_all = np.concatenate(
        [selected_groups[f][2] for f in active_families], axis=1
    )

    # 1. Random Forest
    rf = RandomForestClassifier(
        n_estimators=100,
        max_depth=15,
        min_samples_split=5,
        class_weight="balanced",
        n_jobs=-1,
        random_state=SEED
    )
    rf.fit(Xi_all, y_inner)
    rf_val = rf.predict_proba(Xv_all)[:, 1]
    rf_test = rf.predict_proba(Xt_all)[:, 1]

    # Neural model inputs
    Xi_seq = Xi_all[..., np.newaxis]
    Xv_seq = Xv_all[..., np.newaxis]
    Xt_seq = Xt_all[..., np.newaxis]

    Xi_modal = [selected_groups[f][0] for f in active_families]
    Xv_modal = [selected_groups[f][1] for f in active_families]
    Xt_modal = [selected_groups[f][2] for f in active_families]

    class_ids = np.unique(y_inner)
    class_vals = compute_class_weight(
        class_weight="balanced",
        classes=class_ids,
        y=y_inner
    )
    cw = {int(c): float(w) for c, w in zip(class_ids, class_vals)}

    callbacks = [
        EarlyStopping(
            monitor="val_loss",
            patience=PATIENCE,
            restore_best_weights=True
        )
    ]

    # 2. Hybrid CNN-LSTM
    tf.keras.backend.clear_session()
    tf.random.set_seed(SEED + fold)

    cnn = build_cnn_lstm(Xi_seq.shape[1:])
    cnn.fit(
        Xi_seq, y_inner,
        validation_data=(Xv_seq, y_val),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        class_weight=cw,
        callbacks=callbacks,
        verbose=0
    )
    cnn_val = cnn.predict(Xv_seq, verbose=0).ravel()
    cnn_test = cnn.predict(Xt_seq, verbose=0).ravel()

    # 3. Multimodal Transformer
    tf.keras.backend.clear_session()
    tf.random.set_seed(SEED + 1000 + fold)

    input_dims = [selected_groups[f][0].shape[1] for f in active_families]
    transformer = build_multimodal_transformer(input_dims)

    transformer.fit(
        Xi_modal, y_inner,
        validation_data=(Xv_modal, y_val),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        class_weight=cw,
        callbacks=callbacks,
        verbose=0
    )

    tr_val = transformer.predict(Xv_modal, verbose=0).ravel()
    tr_test = transformer.predict(Xt_modal, verbose=0).ravel()

    # Columns: RF, CNN-LSTM, Transformer
    val_probs = np.column_stack([rf_val, cnn_val, tr_val])
    test_probs = np.column_stack([rf_test, cnn_test, tr_test])

    # 4. Validation-weighted ensemble
    weights = learn_ensemble_weights(val_probs, y_val)
    weighted_test = test_probs @ weights

    # 5. Hard voting (majority vote)
    hard_vote = (
        (test_probs >= 0.5).sum(axis=1) >= 2
    ).astype(int)

    # 6. Soft voting (average probabilities)
    soft_vote = test_probs.mean(axis=1)

    # 7. Stacking, fitted on validation predictions
    if len(np.unique(y_val)) == 2:
        meta = LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=SEED
        )
        meta.fit(val_probs, y_val)
        stacking_test = meta.predict_proba(test_probs)[:, 1]
    else:
        stacking_test = soft_vote.copy()

    results = {
        "RandomForest": rf_test,
        "Hybrid_CNN_LSTM": cnn_test,
        "Multimodal_Transformer": tr_test,
        "Weighted_Ensemble": weighted_test,
        "Hard_Voting": hard_vote.astype(float),
        "Soft_Voting": soft_vote,
        "Stacking": stacking_test
    }

    for model_name, prob in results.items():
        pred = (
            prob.astype(int)
            if model_name == "Hard_Voting"
            else (prob >= 0.5).astype(int)
        )

        metrics = get_metrics(y_test, prob)
        fold_records.append({
            "fold": fold,
            "held_out_subject": test_subject,
            "model": model_name,
            **metrics
        })

        for i in range(len(y_test)):
            all_predictions.append({
                "fold": fold,
                "subject_id": test_subject,
                "model": model_name,
                "true_label": int(y_test[i]),
                "predicted_label": int(pred[i]),
                "PD_probability": float(prob[i])
            })

    weight_records.append({
        "fold": fold,
        "held_out_subject": test_subject,
        "RF_weight": weights[0],
        "CNN_LSTM_weight": weights[1],
        "Transformer_weight": weights[2],
        **{f"{f}_selected": selected_count[f] for f in active_families}
    })

    print("Selected features:", selected_count)
    print("Weights [RF, CNN-LSTM, Transformer]:", np.round(weights, 3))

    del rf, cnn, transformer
    tf.keras.backend.clear_session()

    print("Completed fold", fold)

In [ ]:
#Step 5 — Generate the overall performance comparison
pred_df = pd.DataFrame(all_predictions)
fold_df = pd.DataFrame(fold_records)
weight_df = pd.DataFrame(weight_records)

summary = []

for model_name, part in pred_df.groupby("model"):
    yt = part["true_label"].to_numpy()
    yp = part["predicted_label"].to_numpy()
    pp = part["PD_probability"].to_numpy()

    summary.append({
        "model": model_name,
        **get_metrics(yt, pp)
    })

summary_df = pd.DataFrame(summary).sort_values(
    "roc_auc", ascending=False
)